In [2]:
import pandas as pd
import geopandas as gpd
import networkx as nx
import folium
from shapely.geometry import Point
from pathlib import Path

# load roads with risk
segment_risk = pd.read_csv("../data/processed/segment_risk.csv")
roads = gpd.read_file("../data/external/nyc_roads.gpkg")

roads['physicalid'] = roads['physicalid'].astype(str)
segment_risk['physicalid'] = segment_risk['physicalid'].astype(str)

roads_risk = roads.merge(segment_risk[['physicalid','risk_class','total_csi']], 
                          on='physicalid', how='left')
roads_risk = roads_risk.to_crs("EPSG:3857")

# assign risk weight — safe route avoids high risk segments
risk_weight = {'Low': 1, 'Moderate': 3, 'High': 10}
roads_risk['weight'] = roads_risk['risk_class'].map(risk_weight).fillna(3)

print(f"Roads with risk weights: {len(roads_risk):,}")
print(roads_risk['risk_class'].value_counts())

Roads with risk weights: 122,272
risk_class
Low         44845
Moderate    12436
High         6384
Name: count, dtype: int64


In [ ]:
from shapely.geometry import MultiLineString, LineString
print("Building road network graph...")

G = nx.Graph()

for _, row in roads_risk.iterrows():
    if row.geometry is None:
        continue
    
    # handle both LineString and MultiLineString
    if isinstance(row.geometry, MultiLineString):
        geoms = list(row.geometry.geoms)
    else:
        geoms = [row.geometry]
    
    for geom in geoms:
        coords = list(geom.coords)
        if len(coords) < 2:
            continue
        start = coords[0]
        end = coords[-1]
        length = geom.length
        weight = row['weight'] * length
        
        G.add_edge(start, end,
                   weight=weight,
                   risk_class=row['risk_class'],
                   street=row.get('street_name', ''),
                   geometry=geom)

print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

Building road network graph...


NotImplementedError: Sub-geometries may have coordinate sequences, but multi-part geometries do not

In [ ]:
from shapely.ops import nearest_points
from shapely.geometry import MultiPoint

def find_nearest_node(G, lat, lon):
    # convert to EPSG:3857
    point = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs="EPSG:4326")
    point = point.to_crs("EPSG:3857")
    px, py = point.geometry[0].x, point.geometry[0].y
    
    # find nearest graph node
    nearest = min(G.nodes(), key=lambda n: (n[0]-px)**2 + (n[1]-py)**2)
    return nearest

def safe_route(G, start_lat, start_lon, end_lat, end_lon):
    start_node = find_nearest_node(G, start_lat, start_lon)
    end_node   = find_nearest_node(G, end_lat, end_lon)
    
    try:
        path = nx.shortest_path(G, start_node, end_node, weight='weight')
        return path
    except nx.NetworkXNoPath:
        return None

# Test route — Times Square to Brooklyn Bridge
print("Finding safe route: Times Square → Brooklyn Bridge")
path = safe_route(G, 40.7580, -73.9855, 40.7061, -73.9969)

if path:
    print(f"Route found with {len(path)} waypoints")
else:
    print("No path found")

In [ ]:
if path:
    m = folium.Map(location=[40.7300, -74.0000], zoom_start=13, 
                   tiles='CartoDB positron')
    
    # draw route
    route_coords = [(node[1], node[0]) for node in path]  # lat, lon for folium
    
    # convert back to lat/lon
    route_gdf = gpd.GeoDataFrame(
        geometry=[Point(x, y) for x, y in path], crs="EPSG:3857"
    ).to_crs("EPSG:4326")
    
    route_latlon = [(geom.y, geom.x) for geom in route_gdf.geometry]
    
    folium.PolyLine(
        route_latlon, color='#2980b9', weight=5, opacity=0.9,
        tooltip="Safe Route"
    ).add_to(m)
    
    # start and end markers
    folium.Marker([40.7580, -73.9855], 
                  popup="Start: Times Square",
                  icon=folium.Icon(color='green', icon='play')).add_to(m)
    folium.Marker([40.7061, -73.9969], 
                  popup="End: Brooklyn Bridge",
                  icon=folium.Icon(color='red', icon='stop')).add_to(m)
    
    m.save("../outputs/maps/safe_route.html")
    print("Map saved. Open outputs/maps/safe_route.html")